In [2]:
import requests
import pandas as pd
from datetime import datetime
import calendar
import time

subreddit = "careeradvice"

def fetch_posts_batch(subreddit_name, after_date, before_date, limit=100):
    """
    Fetch a single batch of posts using the Arctic Shift API 
    Params:
    - subreddit_name: str, name of the subreddit
    - after_date: str, 'YYYY-MM-DD' format
    - before_date: str, 'YYYY-MM-DD' format
    - limit: int, number of posts to fetch at once (default 100)
    """
    url = "https://arctic-shift.photon-reddit.com/api/posts/search" # endpoint, see docs @ https://github.com/ArthurHeitmann/arctic_shift/tree/master/api
    
    params = {
        'subreddit': subreddit_name,
        'after': after_date,
        'before': before_date,
        'limit': limit,
        'sort': 'desc'  # newest first
    }
    
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        
        data = response.json()
        if data and 'data' in data:
            return data['data']
        else:
            return []
            
    except Exception as e:
        print(f"Error fetching batch: {e}")
        return []

def collect_posts(subreddit_name, month, year):
    """
    Collect all posts from a month recursively from newest to oldest in batches
    Uses `fetch_posts_batch` to get posts 
    Params:
    - subreddit_name: str, name of the subreddit
    - month: int, month number (1-12)
    - year: int, year (e.g., 2023)
    """
    print(f"Collecting posts from r/{subreddit_name} for {calendar.month_name[month]} {year}")
    
    # convert month to boundaries
    start_date = datetime(year, month, 1)
    if month == 12:
        end_date = datetime(year + 1, 1, 1)
    else:
        end_date = datetime(year, month + 1, 1)
    
    original_after = start_date.strftime('%Y-%m-%d')
    current_before = end_date.strftime('%Y-%m-%d')
    
    all_posts = []
    seen_post_ids = set()
    batch_num = 1

    # recursively collect posts until we reach the end of the month     
    while True:
        print(f"Batch {batch_num}: {original_after} to {current_before}")
        
        # fetch batch of 100 posts
        batch_posts = fetch_posts_batch(subreddit_name, original_after, current_before, 100)
        
        if not batch_posts:
            print(f"No more posts available")
            break
        
        # filter out dupes, add new posts
        new_posts = []
        for post in batch_posts:
            post_id = post.get('id')
            if post_id and post_id not in seen_post_ids:
                seen_post_ids.add(post_id)
                new_posts.append(post)
        
        if not new_posts:
            print(f"No new posts in batch (all duplicates)")
            break
        
        all_posts.extend(new_posts)
        print(f"Added {len(new_posts)} new posts (total: {len(all_posts)})")
        
        if len(batch_posts) < 100:
            print(f" Incomplete batch ({len(batch_posts)} posts) - reached end")
            break
        
        # update timestamp of the oldest post in the fetched batch
        oldest_timestamp = min(post.get('created_utc', 0) for post in batch_posts)
        current_before = datetime.fromtimestamp(oldest_timestamp).strftime('%Y-%m-%d')
        
        # ensure we don't go past the month 
        if current_before <= original_after:
            print(f"Reached date boundary")
            break
        
        batch_num += 1
        time.sleep(0.25)  # rate limiting to 4 req/sec
        
    print(f"Total unique posts collected: {len(all_posts)}")
    return all_posts


def process_posts_to_dataframe(raw_posts, subreddit_name, month, year):
    """
    Convert raw post data to DataFrame for storage and ease of future processing
    Params:
    - raw_posts: list of dicts, raw post data from Arctic Shift
    - subreddit_name: str, name of the subreddit
    - month: int, month number (1-12)
    - year: int, year (e.g., 2023)
    """
    posts_data = []
    for post in raw_posts:
        if not post.get('id') or not post.get('created_utc'):
            continue
        
        # generate permalink properly
        permalink = post.get('permalink', '')
        if permalink and not permalink.startswith('http'):
            # add prefix
            if permalink.startswith('/r/'):
                full_permalink = f"https://www.reddit.com{permalink}"
            else:
                full_permalink = f"https://www.reddit.com/r/{subreddit_name}/comments/{post.get('id', '')}"
        else:
            full_permalink = permalink

        # ignore fully external posts (no selftext)
        if not post.get('selftext'):
            continue

        # structure fields from the post into DataFrame            
        posts_data.append({
            'post_id': post.get('id'),
            'title': post.get('title', ''),
            'text': post.get('selftext', ''),
            'author': post.get('author', '[deleted]'),
            'score': post.get('score', 0),
            'upvote_ratio': post.get('upvote_ratio', 0),
            'num_comments': post.get('num_comments', 0),
            'created_utc': post.get('created_utc'),
            'created_datetime': datetime.fromtimestamp(post.get('created_utc', 0)),
            'subreddit': subreddit_name,
            'permalink': full_permalink,  
            'month': month,
            'year': year
        })
    
    return pd.DataFrame(posts_data)

def get_all_posts_arctic_shift(subreddit_name, month, year):
    """
    get ALL posts from a subreddit in a specific month using Arctic Shift API
    Params:
    - subreddit_name: str, name of the subreddit
    - month: int, month number (1-12)
    - year: int, year (e.g., 2023)
    """    
    raw_posts = collect_posts(subreddit_name, month, year)
    
    if not raw_posts:
        print("No posts collected")
        return pd.DataFrame()
    
    df = process_posts_to_dataframe(raw_posts, subreddit_name, month, year)
    
    if df.empty:
        print("No valid posts processed")
        return pd.DataFrame()
    
    print(f"Collected {len(raw_posts)} total posts from r/{subreddit_name} for {calendar.month_name[month]} {year}")
    
    return df

for year in [2023, 2024, 2025]: 
    start_month = 8 if year == 2023 else 1
    end_month = 8 if year == 2025 else 12

    for month in range(start_month, end_month + 1):
        all_posts = get_all_posts_arctic_shift(subreddit, month, year)
        all_posts.to_csv(f"updated_data/raw/information/careeradvice/{year}_{month:02d}.csv", index=False)

Batch 1: 2023-08-01 to 2023-09-01
Added 100 new posts (total: 100)
Batch 2: 2023-08-01 to 2023-08-30
Added 100 new posts (total: 200)
Batch 3: 2023-08-01 to 2023-08-28
Added 100 new posts (total: 300)
Batch 4: 2023-08-01 to 2023-08-26
Added 100 new posts (total: 400)
Batch 5: 2023-08-01 to 2023-08-24
Added 100 new posts (total: 500)
Batch 6: 2023-08-01 to 2023-08-22
Added 100 new posts (total: 600)
Batch 7: 2023-08-01 to 2023-08-20
Added 100 new posts (total: 700)
Batch 8: 2023-08-01 to 2023-08-18
Added 100 new posts (total: 800)
Batch 9: 2023-08-01 to 2023-08-16
Added 100 new posts (total: 900)
Batch 10: 2023-08-01 to 2023-08-14
Added 100 new posts (total: 1000)
Batch 11: 2023-08-01 to 2023-08-12
Added 100 new posts (total: 1100)
Batch 12: 2023-08-01 to 2023-08-10
Added 100 new posts (total: 1200)
Batch 13: 2023-08-01 to 2023-08-08
Added 100 new posts (total: 1300)
Batch 14: 2023-08-01 to 2023-08-06
Added 100 new posts (total: 1400)
Batch 15: 2023-08-01 to 2023-08-04
Added 100 new pos

In [42]:
import pandas as pd, glob, os, re

FOLDER = "./updated_data/raw/careeradvice"

# 1) Expected schema + simple renames (map common variants -> standard names)
USECOLS = [
    "post_id","title","selftext","author","score","upvote_ratio",
    "num_comments","created_utc","created_datetime","subreddit",
    "permalink","month","year"
]
RENAME_MAP = {
    "id": "post_id",
    "text": "selftext",
    "created": "created_utc",
    "created_at": "created_datetime",
}
NUMERIC_COLS = ["score", "upvote_ratio", "num_comments"]

files = sorted(glob.glob(os.path.join(FOLDER, "*.csv")))
print(f"[info] files found: {len(files)}")

dfs = []
for f in files:
    try:
        df = pd.read_csv(f)
        # normalize header
        df.columns = [c.strip() for c in df.columns]
        # rename common variants
        df = df.rename(columns={k: v for k, v in RENAME_MAP.items() if k in df.columns})

        # keep only known columns but don't drop all if missing; add missing as NaN
        df = df.reindex(columns=USECOLS)

        # coerce numerics (safe)
        for c in NUMERIC_COLS:
            if c in df.columns:
                # clean strings like "1,234" or "12 comments"
                s = (
                    df[c].astype(str)
                    .str.replace(",", "", regex=False)
                    .str.extract(r"(^-?\d+\.?\d*)", expand=False)
                )
                df[c] = pd.to_numeric(s, errors="coerce")

        # fix upvote_ratio if 0–100
        if "upvote_ratio" in df.columns and df["upvote_ratio"].notna().any():
            ur = df["upvote_ratio"]
            if pd.to_numeric(ur, errors="coerce").max(skipna=True) > 1.0:
                df["upvote_ratio"] = df["upvote_ratio"] / 100.0
            df["upvote_ratio"] = df["upvote_ratio"].clip(0, 1)

        dfs.append(df)
    except Exception as e:
        print(f"[warn] {os.path.basename(f)} -> {e}")

if not dfs:
    raise SystemExit("No readable CSVs; check warnings above.")

df_raw = pd.concat(dfs, ignore_index=True, sort=False)

# final NA handling for numerics
for c in NUMERIC_COLS:
    if c in df_raw.columns:
        df_raw[c] = pd.to_numeric(df_raw[c], errors="coerce").fillna(0)

print("[info] concatenated shape:", df_raw.shape)
print(df_raw[["score","upvote_ratio","num_comments"]].dtypes)


[info] files found: 25
[info] concatenated shape: (38735, 13)
score           float64
upvote_ratio    float64
num_comments    float64
dtype: object


In [44]:
# CLEAN THE DATA
import re

# --- choose text column ---
TEXT_COL = next(c for c in ["selftext", "body", "text"] if c in df_raw.columns)

# --- drop empties / placeholders ---
df_raw[TEXT_COL] = df_raw[TEXT_COL].fillna("").astype(str).str.strip()
PLACEHOLDERS = r"(?i)^\s*(\[deleted\]|\[removed\]|deleted|removed|n/?a|null|none)\s*$"
df_raw = df_raw[df_raw[TEXT_COL].ne("") & ~df_raw[TEXT_COL].str.match(PLACEHOLDERS)]

# --- combine title + text (if applicable) ---
if "title" in df_raw.columns:
    df_raw["text_full"] = (df_raw["title"].fillna("").astype(str) + " " + df_raw[TEXT_COL]).str.strip()
else:
    df_raw["text_full"] = df_raw[TEXT_COL]

# --- clean text lightly ---
def clean_text(s: str) -> str:
    s = re.sub(r"http\S+|www\.\S+", " ", s)
    s = re.sub(r"&amp;#x200B;|&nbsp;|&amp;", " ", s)
    s = re.sub(r"`{3}.*?`{3}", " ", s, flags=re.S)
    s = re.sub(r"`[^`]*`", " ", s)
    s = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", s)
    s = re.sub(r">+\s.*", " ", s)
    s = re.sub(r"[\r\n\t]+", " ", s)
    s = re.sub(r"\s{2,}", " ", s).strip()
    return s

df_raw["text_full"] = df_raw["text_full"].map(clean_text)

# --- drop very short / duplicate content ---
df_raw = df_raw[df_raw["text_full"].str.len() >= 10]
df_raw = df_raw[df_raw["text_full"].str.count(r"\w") >= 5]
if "id" in df_raw.columns:
    df_raw = df_raw.drop_duplicates(subset=["id"])
df_raw = df_raw.drop_duplicates(subset=["text_full"])

In [46]:
df.head()

,post_id,title,selftext,author,score,upvote_ratio,num_comments,created_utc,created_datetime,subreddit,permalink,month,year
0,1n598hl,NYS Labor Law Violation? Fired for leaving wor...,TL;DR: Fired after going to the dentist \n\nIt...,parabolic86,1,0.56,5,1756682629,2025-08-31 18:23:49,careeradvice,https://www.reddit.com/r/careeradvice/comments...,8,2025
1,1n5958y,"Looking for advice, need to vent a little",So I've all but been fired (union is arguing b...,CatLess186,1,1.00,2,1756682375,2025-08-31 18:19:35,careeradvice,https://www.reddit.com/r/careeradvice/comments...,8,2025
2,1n57oe2,What jobs have you found as a Visual Communica...,Hi! \n\nI am currently majoring in VCD in the ...,CriticismStock9268,1,1.00,2,1756678392,2025-08-31 17:13:12,careeradvice,https://www.reddit.com/r/careeradvice/comments...,8,2025
3,1n57bzi,My parents want me to pursue ACCA and i'm not ...,I'm genuinely so confused about what i should ...,highonnuggetss,0,0.50,0,1756677498,2025-08-31 16:58:18,careeradvice,https://www.reddit.com/r/careeradvice/comments...,8,2025
4,1n576l4,Feeling unsure about love of professional,I’ve been a software engineer for about 3 year...,InevitableNo5158,3,1.00,1,1756677097,2025-08-31 16:51:37,careeradvice,https://www.reddit.com/r/careeradvice/comments...,8,2025


In [49]:
columns_to_show = ["title", ", "score", "upvote_ratio", "num_comments"]

df_raw["engagement_metric"] = (
    df_raw["score"].astype(float)
    * df_raw["upvote_ratio"].astype(float)
    * (df_raw["num_comments"].astype(float) + 1)
)

In [53]:
output_folder = "updated_data/top_and_bottom_15"
os.makedirs(output_folder, exist_ok=True)

top_15_posts = df_raw.sort_values(by="engagement_metric", ascending=False).head(15)
top_path = os.path.join(output_folder, "top_15_posts_full.csv")
top_15_posts.to_csv(top_path, index=False)

bottom_15_posts = df_raw.sort_values(by="engagement_metric", ascending=True).head(15)
bottom_path = os.path.join(output_folder, "bottom_15_posts_full.csv")
bottom_15_posts.to_csv(bottom_path, index=False)